In [ ]:
!pip install -qU langchain langchain-core langchain-groq langchain-chroma langchain-google-genai langchain-community pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.5/329.5 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.3/137.3 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 59.3 MB/s eta 0:00:00


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load the document
file_path = "/content/Rohit Das.pdf"
loader = PyPDFLoader(file_path)
pages = loader.load()

# Split the document into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=150)
docs = text_splitter.split_documents(pages)

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from google.colab import userdata

# Create embeddings
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=userdata.get('GEMINI_API_KEY'))

# Initialize and populate the Vector Store
vector_store = Chroma(
    collection_name="resume_bot_collection",
    embedding_function=embeddings,
)
vector_store.add_documents(documents=docs)

['43aab4e1-ee0c-4d3b-bd4a-ecc7da8b8600',
 'd01edab1-feae-4028-bc85-f3ad809388fb',
 'e6556b2b-a3ee-4894-930f-ada63c7480b1',
 'b66b98dd-d5f0-41f0-a2a8-ab2828e1580d',
 '36dcc003-60d3-425a-ad4c-a99b1eae5f20',
 '2bcdb99f-6a04-476c-9f6a-f0c7d7341158',
 'a806bd07-25fa-4327-b2fd-9a43022118ce',
 'a128c251-ee64-4719-9b3a-faa4c6d0e307',
 '59e9d25a-bb45-48f2-8e12-b4e69fbcdb20',
 'b6d45eca-b217-4247-be19-57581646879c',
 'a795e11c-5439-4565-89e5-3cf4aae5853f',
 '50313200-8ea5-4610-aa65-18e6c9426250',
 '98a2623c-a472-4e19-bcc1-bdcae4294e16',
 '8a98285f-cdd9-4b2f-98d3-cdb76390a947',
 'd978aa26-eb38-40e8-b3ec-4c65a67f5e2d',
 '16dbb63f-5bae-428e-be96-54ef22301b90',
 'bd65d7ed-80e2-47d8-bd94-7b82348afadd']

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [ ]:
from langchain_core.tools import tool

@tool
def get_resume_data(query: str) -> str:
  """
  Get information about Rohit Das's background experience and projects, from the resume.
  """
  return retriever.invoke(query)

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(api_key=userdata.get('GROQ_API_KEY'),
               model="moonshotai/kimi-k2-instruct-0905",
               temperature=0)

# Bind the tool to the LLM
llm_with_tools = llm.bind_tools([get_resume_data])

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage

messages = []

system_msg = SystemMessage("""
You are an assistant for Rohit Das. If you don’t know an answer, call the tool 'get_resume_data' with the question to retrieve info from the resume.
Answer ONLY with info retrieved from the tool.
""")

messages.append(system_msg)

while True:
    user_input = input("\nEnter your query: ").strip()
    if user_input.lower() in {"exit", "close", "quit"}:
        break

    messages.append(HumanMessage(content=user_input))

    combined_message = messages
    ai_message = llm_with_tools.invoke(combined_message)
    messages.append(ai_message)

    if ai_message.tool_calls:
        # Tool call detected, execute the tool call(s)
        for call in ai_message.tool_calls:
            tool_args = call.get("args") or call.get("arguments") or {}

            # Call the get_resume_data tool using the correct args
            tool_output = str(get_resume_data.invoke(tool_args))

            # Append the tool output message with the tool call ID
            messages.append(ToolMessage(tool_call_id=call["id"], content=tool_output))

        # Re-invoke the LLM with the updated messages to get the final answer
        final_response = llm_with_tools.invoke(messages)
        messages.append(final_response)

        print("\nAnswer:")
        print(final_response.content)

    else:
        # No tool call, provide direct answer
        print("\nAnswer:")
        print(ai_message.content)


Enter your query: What is your  name?

Answer:
My name is Rohit Das.

Enter your query: Can you tell me about your projects?

Answer:
Here are some of my key projects:

- **ESG Analysis with NLP**: Applied advanced Natural Language Processing techniques using BERT and SpaCy to streamline the analysis of Environmental, Social, and Governance (ESG) criteria, enhancing the assessment of corporate sustainability and ethical impacts.

- **Eye Tracking Research**: Developed methodologies to benchmark eye tracking predictions in videos and images with over 94% accuracy, evaluating ViNET and UNISAL models for predicting viewer focus areas.

- **Retail Data Analysis**: Used Time-Series Analysis and TimeGAN to analyze Retail Scanner data across the United States, uncovering consumption trends that improved itemized sales by 62%.

- **Machine Learning Poster Generation**: Led a university project to produce high-quality poster images through advanced machine learning techniques.

- **ETL Pipelin